In [1]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

In [2]:
import sys
sys.path.append("..") 

In [3]:
from processing import load_UCI_dataset,extract_all_features,extract_features_from_dataset, select_f_test, select_mrmr, select_reliefF, save_feature_set

In [4]:
X_train, y_train, X_val, y_val, X_test, y_test = load_UCI_dataset(WINDOW_SIZE= 625, STEP_SIZE= 312)

Total recordings: 12000
Train recordings: 9600
Validation recordings: 1200
Test recordings: 1200


100%|██████████| 9600/9600 [03:14<00:00, 49.46it/s]


Skipped recordings: 3


100%|██████████| 1200/1200 [00:25<00:00, 46.70it/s]


Skipped recordings: 1


100%|██████████| 1200/1200 [00:25<00:00, 47.18it/s]


Skipped recordings: 0


In [5]:
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print("X_train min:", X_train.min())
print("X_train max:", X_train.max())

X_train shape: (839014, 1, 625), y_train shape: (839014, 2)
X_train min: 0.0
X_train max: 1.0


## Feature Exraction - Non-fiducial features ##

We'll extract 19 features

## Feature Exraction - Non-fiducial features ##

We'll extract 19 features

In [6]:
test_window = X_train[0, 0, :]

features = extract_all_features(
    test_window,
    fs=125
)

print("Number of features:", len(features))

print(features)

Number of features: 57
{'ppg_mean': np.float64(0.5528637123954845), 'ppg_median': np.float64(0.5925931433156838), 'ppg_std': np.float64(0.2905845371580045), 'ppg_variance': np.float64(0.08443937323533167), 'ppg_iqr': np.float64(0.5234703093409055), 'ppg_skewness': np.float64(-0.22736716400990045), 'ppg_kurtosis': np.float64(-1.2339547439481824), 'ppg_zero_crossing_rate': np.float64(0.0), 'ppg_shannon_entropy': np.float64(5.574628051267654), 'ppg_energy_mean': np.float64(0.3900976577190485), 'ppg_energy_variance': np.float64(0.09636289363250157), 'ppg_energy_skewness': np.float64(0.3190031184262097), 'ppg_energy_kurtosis': np.float64(-1.2737042691425176), 'ppg_energy_iqr': np.float64(0.5817711450048391), 'ppg_kte_mean': np.float64(0.0015399260317650507), 'ppg_kte_variance': np.float64(4.096121882933001e-06), 'ppg_kte_skewness': np.float64(1.1982202259142651), 'ppg_kte_kurtosis': np.float64(1.2361426030677016), 'ppg_kte_iqr': np.float64(0.002613196788427148), 'vpg_mean': np.float64(-0.00

In [7]:
df_train = extract_features_from_dataset(
    X_train,
    y_train
)

df_val = extract_features_from_dataset(
    X_val,
    y_val
)

df_test = extract_features_from_dataset(
    X_test,
    y_test
)

In [8]:
print("Train:", df_train.shape)
print("Validation:", df_val.shape)
print("Test:", df_test.shape)

Train: (839014, 59)
Validation: (105924, 59)
Test: (106401, 59)


## Feature Selection ##

We'll reduce the 57 features to 15 using 3 different methods

In [9]:
feature_columns = [
    col for col in df_train.columns
    if col not in ["SBP", "DBP"]
]

X_train = df_train[feature_columns]
X_val = df_val[feature_columns]
X_test = df_test[feature_columns]

y_train_sbp = df_train["SBP"]
y_train_dbp = df_train["DBP"]

y_val_sbp = df_val["SBP"]
y_val_dbp = df_val["DBP"]

y_test_sbp = df_test["SBP"]
y_test_dbp = df_test["DBP"]

## F-Test ##

In [10]:
# SBP
f_sbp_features, f_sbp_results = select_f_test(
    X_train,
    y_train_sbp,
    k=15
)
print("F-test selected features for SBP:")

for i, feature in enumerate(f_sbp_features, 1):
    print(i, feature)

print(f_sbp_results)

F-test selected features for SBP:
1 vpg_iqr
2 vpg_skewness
3 apg_zero_crossing_rate
4 vpg_energy_iqr
5 apg_energy_iqr
6 ppg_iqr
7 apg_skewness
8 vpg_shannon_entropy
9 apg_shannon_entropy
10 vpg_kte_iqr
11 apg_iqr
12 vpg_energy_skewness
13 ppg_kte_iqr
14 vpg_kurtosis
15 ppg_energy_iqr
                   feature       F_score        p_value
0                  vpg_iqr  38566.996172   0.000000e+00
1             vpg_skewness  32931.184696   0.000000e+00
2   apg_zero_crossing_rate  28915.381222   0.000000e+00
3           vpg_energy_iqr  28660.845053   0.000000e+00
4           apg_energy_iqr  19764.955655   0.000000e+00
5                  ppg_iqr  19241.073903   0.000000e+00
6             apg_skewness  17441.900091   0.000000e+00
7      vpg_shannon_entropy  17218.369896   0.000000e+00
8      apg_shannon_entropy  15590.472227   0.000000e+00
9              vpg_kte_iqr  15487.059970   0.000000e+00
10                 apg_iqr  14594.953986   0.000000e+00
11     vpg_energy_skewness  12895.053353   

In [11]:
# DBP
f_dbp_features, f_dbp_results = select_f_test(
    X_train,
    y_train_dbp,
    k=15
)

print("\nF-test selected features for DBP:")

for i, feature in enumerate(f_dbp_features, 1):
    print(i, feature)

print(f_dbp_results)


F-test selected features for DBP:
1 apg_iqr
2 ppg_mean
3 vpg_median
4 apg_kte_iqr
5 apg_kte_mean
6 ppg_median
7 apg_std
8 vpg_kte_mean
9 apg_median
10 apg_variance
11 apg_energy_mean
12 ppg_skewness
13 ppg_shannon_entropy
14 ppg_energy_skewness
15 ppg_kte_iqr
                   feature       F_score        p_value
0                  apg_iqr  17832.321190   0.000000e+00
1                 ppg_mean  17599.497893   0.000000e+00
2               vpg_median  17424.177560   0.000000e+00
3              apg_kte_iqr  17177.427995   0.000000e+00
4             apg_kte_mean  15523.033749   0.000000e+00
5               ppg_median  14630.717801   0.000000e+00
6                  apg_std  13948.089957   0.000000e+00
7             vpg_kte_mean  13467.699245   0.000000e+00
8               apg_median  13428.834358   0.000000e+00
9             apg_variance  11672.119037   0.000000e+00
10         apg_energy_mean  11657.109561   0.000000e+00
11            ppg_skewness  10970.982333   0.000000e+00
12     ppg_

## mRMR ##

In [12]:
mrmr_sbp_features, mrmr_sbp_results = select_mrmr(
    X_train,
    y_train_sbp,
    k=15
)

print("mRMR selected features for SBP:")

for i, feature in enumerate(
    mrmr_sbp_features,
    1
):
    print(i, feature)

print(mrmr_sbp_results)

mRMR: using 50000 samples for feature selection
mRMR selected features for SBP:
1 vpg_iqr
2 ppg_zero_crossing_rate
3 apg_energy_variance
4 ppg_energy_iqr
5 vpg_mean
6 ppg_median
7 apg_energy_skewness
8 apg_mean
9 apg_zero_crossing_rate
10 ppg_std
11 apg_median
12 vpg_median
13 ppg_kte_kurtosis
14 apg_energy_iqr
15 vpg_skewness
                   feature  relevance_MI  avg_redundancy  mRMR_score  \
0                  vpg_iqr      0.058743        0.000000    0.058743   
1   ppg_zero_crossing_rate      0.000072        0.000000    0.000072   
2      apg_energy_variance      0.041187        0.024074    0.017113   
3           ppg_energy_iqr      0.035789        0.031847    0.003941   
4                 vpg_mean      0.000966        0.006177   -0.005211   
5               ppg_median      0.051202        0.064341   -0.013139   
6      apg_energy_skewness      0.050296        0.070939   -0.020643   
7                 apg_mean      0.006254        0.031011   -0.024757   
8   apg_zero_crossing_r

In [13]:
mrmr_dbp_features, mrmr_dbp_results = select_mrmr(
    X_train,
    y_train_dbp,
    k=15
)

print("\nmRMR selected features for DBP:")

for i, feature in enumerate(
    mrmr_dbp_features,
    1
):
    print(i, feature)


print(mrmr_dbp_results)

mRMR: using 50000 samples for feature selection

mRMR selected features for DBP:
1 ppg_mean
2 apg_kte_iqr
3 ppg_zero_crossing_rate
4 ppg_std
5 apg_mean
6 apg_zero_crossing_rate
7 ppg_kte_kurtosis
8 vpg_mean
9 apg_median
10 vpg_median
11 apg_energy_variance
12 ppg_kte_iqr
13 apg_skewness
14 vpg_iqr
15 ppg_iqr
                   feature  relevance_MI  avg_redundancy  mRMR_score  \
0                 ppg_mean      0.041781        0.000000    0.041781   
1              apg_kte_iqr      0.041675        0.030481    0.011194   
2   ppg_zero_crossing_rate      0.000295        0.000000    0.000295   
3                  ppg_std      0.013857        0.014441   -0.000585   
4                 apg_mean      0.007193        0.009771   -0.002578   
5   apg_zero_crossing_rate      0.018642        0.022886   -0.004244   
6         ppg_kte_kurtosis      0.014719        0.026688   -0.011968   
7                 vpg_mean      0.000316        0.025439   -0.025123   
8               apg_median      0.019393  

## ReliefF ##

In [14]:
relief_sbp_features, relief_sbp_results = select_reliefF(
    X_train,
    y_train_sbp,
    k=15
)

print("ReliefF selected features for SBP:")

for i, feature in enumerate(
    relief_sbp_features,
    1
):
    print(i, feature)


print(relief_sbp_results)

RReliefF: using 50000 samples for feature selection
ReliefF selected features for SBP:
1 vpg_mean
2 apg_mean
3 apg_zero_crossing_rate
4 apg_median
5 vpg_median
6 apg_skewness
7 ppg_energy_variance
8 vpg_skewness
9 vpg_shannon_entropy
10 vpg_zero_crossing_rate
11 ppg_variance
12 ppg_std
13 ppg_kte_iqr
14 ppg_shannon_entropy
15 vpg_iqr
                   feature  RReliefF_score
0                 vpg_mean        0.101885
1                 apg_mean        0.096733
2   apg_zero_crossing_rate        0.088103
3               apg_median        0.075397
4               vpg_median        0.069665
5             apg_skewness        0.068588
6      ppg_energy_variance        0.067500
7             vpg_skewness        0.065170
8      vpg_shannon_entropy        0.063872
9   vpg_zero_crossing_rate        0.063159
10            ppg_variance        0.061424
11                 ppg_std        0.060859
12             ppg_kte_iqr        0.060288
13     ppg_shannon_entropy        0.060037
14                 

In [15]:
relief_dbp_features, relief_dbp_results = select_reliefF(
    X_train,
    y_train_dbp,
    k=15
)

print("\nReliefF selected features for DBP:")

for i, feature in enumerate(
    relief_dbp_features,
    1
):
    print(i, feature)
    
print(relief_dbp_results)

RReliefF: using 50000 samples for feature selection

ReliefF selected features for DBP:
1 vpg_mean
2 apg_mean
3 apg_zero_crossing_rate
4 apg_median
5 vpg_median
6 ppg_energy_variance
7 apg_skewness
8 vpg_skewness
9 vpg_shannon_entropy
10 vpg_zero_crossing_rate
11 ppg_variance
12 ppg_std
13 ppg_kte_iqr
14 vpg_iqr
15 ppg_shannon_entropy
                   feature  RReliefF_score
0                 vpg_mean        0.065935
1                 apg_mean        0.063173
2   apg_zero_crossing_rate        0.056150
3               apg_median        0.050036
4               vpg_median        0.045271
5      ppg_energy_variance        0.044040
6             apg_skewness        0.043902
7             vpg_skewness        0.042096
8      vpg_shannon_entropy        0.040711
9   vpg_zero_crossing_rate        0.040195
10            ppg_variance        0.040013
11                 ppg_std        0.039727
12             ppg_kte_iqr        0.039171
13                 vpg_iqr        0.038826
14     ppg_shannon

In [16]:
selection_summary = {

    "SBP_F_test": f_sbp_features,

    "SBP_mRMR": mrmr_sbp_features,

    "SBP_ReliefF": relief_sbp_features,

    "DBP_F_test": f_dbp_features,

    "DBP_mRMR": mrmr_dbp_features,

    "DBP_ReliefF": relief_dbp_features
}

In [17]:
for name, features in selection_summary.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    for i, feature in enumerate(features, 1):
        print(f"{i:2d}. {feature}")


SBP_F_test
 1. vpg_iqr
 2. vpg_skewness
 3. apg_zero_crossing_rate
 4. vpg_energy_iqr
 5. apg_energy_iqr
 6. ppg_iqr
 7. apg_skewness
 8. vpg_shannon_entropy
 9. apg_shannon_entropy
10. vpg_kte_iqr
11. apg_iqr
12. vpg_energy_skewness
13. ppg_kte_iqr
14. vpg_kurtosis
15. ppg_energy_iqr

SBP_mRMR
 1. vpg_iqr
 2. ppg_zero_crossing_rate
 3. apg_energy_variance
 4. ppg_energy_iqr
 5. vpg_mean
 6. ppg_median
 7. apg_energy_skewness
 8. apg_mean
 9. apg_zero_crossing_rate
10. ppg_std
11. apg_median
12. vpg_median
13. ppg_kte_kurtosis
14. apg_energy_iqr
15. vpg_skewness

SBP_ReliefF
 1. vpg_mean
 2. apg_mean
 3. apg_zero_crossing_rate
 4. apg_median
 5. vpg_median
 6. apg_skewness
 7. ppg_energy_variance
 8. vpg_skewness
 9. vpg_shannon_entropy
10. vpg_zero_crossing_rate
11. ppg_variance
12. ppg_std
13. ppg_kte_iqr
14. ppg_shannon_entropy
15. vpg_iqr

DBP_F_test
 1. apg_iqr
 2. ppg_mean
 3. vpg_median
 4. apg_kte_iqr
 5. apg_kte_mean
 6. ppg_median
 7. apg_std
 8. vpg_kte_mean
 9. apg_median


## Saving ##

In [18]:
save_feature_set(
    name="SBP_FTEST",
    selected_features=f_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="SBP_MRMR",
    selected_features=mrmr_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="SBP_RELIEFF",
    selected_features=relief_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

Processing: SBP_FTEST
Number of selected features: 15
Train shape: (839014, 16)
Val shape:   (105924, 16)
Test shape:  (106401, 16)

Saved:
data/SBP_FTEST_train.csv
data/SBP_FTEST_val.csv
data/SBP_FTEST_test.csv
data/SBP_FTEST_preprocessing.pkl

Processing: SBP_MRMR
Number of selected features: 15
Train shape: (839014, 16)
Val shape:   (105924, 16)
Test shape:  (106401, 16)

Saved:
data/SBP_MRMR_train.csv
data/SBP_MRMR_val.csv
data/SBP_MRMR_test.csv
data/SBP_MRMR_preprocessing.pkl

Processing: SBP_RELIEFF
Number of selected features: 15
Train shape: (839014, 16)
Val shape:   (105924, 16)
Test shape:  (106401, 16)

Saved:
data/SBP_RELIEFF_train.csv
data/SBP_RELIEFF_val.csv
data/SBP_RELIEFF_test.csv
data/SBP_RELIEFF_preprocessing.pkl



In [19]:
save_feature_set(
    name="DBP_FTEST",
    selected_features=f_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="DBP_MRMR",
    selected_features=mrmr_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="DBP_RELIEFF",
    selected_features=relief_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

Processing: DBP_FTEST
Number of selected features: 15
Train shape: (839014, 16)
Val shape:   (105924, 16)
Test shape:  (106401, 16)

Saved:
data/DBP_FTEST_train.csv
data/DBP_FTEST_val.csv
data/DBP_FTEST_test.csv
data/DBP_FTEST_preprocessing.pkl

Processing: DBP_MRMR
Number of selected features: 15
Train shape: (839014, 16)
Val shape:   (105924, 16)
Test shape:  (106401, 16)

Saved:
data/DBP_MRMR_train.csv
data/DBP_MRMR_val.csv
data/DBP_MRMR_test.csv
data/DBP_MRMR_preprocessing.pkl

Processing: DBP_RELIEFF
Number of selected features: 15
Train shape: (839014, 16)
Val shape:   (105924, 16)
Test shape:  (106401, 16)

Saved:
data/DBP_RELIEFF_train.csv
data/DBP_RELIEFF_val.csv
data/DBP_RELIEFF_test.csv
data/DBP_RELIEFF_preprocessing.pkl

